In [2]:
import akshare as ak
import pandas as pd
import numpy as np

from tenacity import retry, stop_after_attempt, wait_fixed, wait_random, retry_if_exception_type
import logging

In [3]:
# 配置日志，以便知道发生了重试
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# http visit strategy
# --- 定义重试策略 ---
# 1. stop_after_attempt(3): 最多尝试 3 次
# 2. wait_fixed(2) + wait_random(0, 2): 每次失败后等待 2~4 秒 (防止高频请求被封)
# 3. retry_if_exception_type(Exception): 针对所有异常重试（也可以指定 ConnectionError）
common_retry_strategy = retry(
    stop=stop_after_attempt(3),
    wait=wait_fixed(2) + wait_random(0, 2),
    retry=retry_if_exception_type(Exception),
    reraise=True,  # 如果3次都失败，抛出异常供上层处理
    before_sleep=lambda retry_state: logger.warning(f"请求失败，准备进行第 {retry_state.attempt_number} 次重试...")
)


In [4]:
# Meta data get
class FundDataSource:
    @common_retry_strategy
    def get_industry_board_list(self) -> pd.DataFrame:
        """
        获取东方财富-行业板块列表（实时行情）
        对应元数据：板块名称, 板块代码, 涨跌幅, 总市值, 换手率
        接口: ak.stock_board_industry_name_em()
        """
        logger.info("正在调用 ak.stock_board_industry_name_em 获取行业板块数据...")
        
        try:
            # 1. 调用 AkShare 接口
            df = ak.stock_board_industry_name_em()
            
            # 2. 基础数据校验
            if df is None or df.empty:
                logger.warning("接口返回数据为空！可能休市或接口异常。")
                # 可以在这里抛出异常触发 retry，或者返回空 DataFrame
                raise ValueError("AkShare 接口返回空数据")
            
            # 3. 数据清洗 (清洗为你的元数据标准)
            # 原始列名通常包含：['排名', '板块名称', '板块代码', '最新价', '涨跌额', '涨跌幅', '总市值', '换手率', '上涨家数', '下跌家数', '领涨股票', '领涨股票-涨跌幅']
            
            # 强制转换数值列，防止因为包含 "%" 或 "-" 导致变成字符串
            if '涨跌幅' in df.columns:
                 # 部分接口可能返回数值，也可能返回字符串，做个防御性转换
                df['涨跌幅'] = pd.to_numeric(df['涨跌幅'], errors='coerce')

            logger.info(f"成功获取 {len(df)} 个行业板块数据。")
            return df
            
        except Exception as e:
            logger.error(f"获取行业板块数据失败: {e}")
            raise e  # 抛出异常以便触发 @common_retry_strategy
    

In [6]:
# Interface test
print("-" * 30)
print("开始调试 get_industry_board_list ...")

source = FundDataSource()

try:
    # 1. 执行函数
    df_industry = source.get_industry_board_list()
    
    # 2. 打印元数据结构 (Columns)
    print("\n[元数据结构检查]:")
    print(f"列名列表: {df_industry.columns.tolist()}")
    
    # 3. 打印样例数据
    print("\n[数据样例 - 前 5 行]:")
    # 选取几个核心字段打印，方便肉眼核对
    display_cols = ['板块名称', '板块代码', '最新价', '涨跌幅', '换手率', '领涨股票']
    # 确保列名存在再打印，防止报错
    existing_cols = [c for c in display_cols if c in df_industry.columns]
    df_industry[existing_cols].head().to_markdown(index=False)
    
    # 4. 验证关键数据有效性
    print("\n[数据质量验证]:")
    top_gainer = df_industry.sort_values(by='涨跌幅', ascending=False).iloc[0]
    print(f"今日最强板块: {top_gainer['板块名称']} (涨幅: {top_gainer['涨跌幅']}%)")
    print(f"领涨龙头股: {top_gainer['领涨股票']}")

except Exception as e:
    print(f"\n[调试失败]: {e}")

INFO:__main__:正在调用 ak.stock_board_industry_name_em 获取行业板块数据...


------------------------------
开始调试 get_industry_board_list ...


0it [00:00, ?it/s]

INFO:__main__:成功获取 86 个行业板块数据。



[元数据结构检查]:
列名列表: ['排名', '板块名称', '板块代码', '最新价', '涨跌额', '涨跌幅', '总市值', '换手率', '上涨家数', '下跌家数', '领涨股票', '领涨股票-涨跌幅']

[数据样例 - 前 5 行]:

[数据质量验证]:
今日最强板块: 商业百货 (涨幅: 3.01%)
领涨龙头股: 百大集团
